# RAG: chat with the reviews
Embeds the 294 distinct review comments, indexes them with Databricks Vector Search, retrieves the closest comments to a question, and answers from them with the sources shown.

## Embeddings table
One row per distinct comment, with its sentiment, topic, how many reviews use it, and a 1024-number vector from the `databricks-gte-large-en` embedding model.

In [0]:
%sql
CREATE OR REPLACE TABLE zomato_ai.gold.comment_embeddings AS
SELECT
  row_number() OVER (ORDER BY e.comment) AS comment_id,
  e.comment, e.sentiment, e.topic,
  c.review_count, c.avg_rating,
  ai_query('databricks-gte-large-en', e.comment) AS embedding
FROM zomato_ai.gold.comment_enrichment e
JOIN (
  SELECT comment, count(*) AS review_count, round(avg(rating), 2) AS avg_rating
  FROM zomato_ai.gold.fact_reviews
  GROUP BY comment
) c ON e.comment = c.comment

num_affected_rows,num_inserted_rows


## Vector Search index
Change data feed lets the index follow the table. The endpoint and index are created only if they don't already exist; otherwise the index is re-synced.

In [0]:
%sql
ALTER TABLE zomato_ai.gold.comment_embeddings
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, VectorIndexType, DeltaSyncVectorIndexSpecRequest,
    EmbeddingVectorColumn, PipelineType)

w = WorkspaceClient()
ENDPOINT = "zomato_vs_endpoint"
INDEX = "zomato_ai.gold.comment_index"

if ENDPOINT not in [e.name for e in w.vector_search_endpoints.list_endpoints()]:
    w.vector_search_endpoints.create_endpoint_and_wait(
        name=ENDPOINT, endpoint_type=EndpointType.STANDARD)
else:
    print("endpoint already exists")

if INDEX not in [i.name for i in w.vector_search_indexes.list_indexes(endpoint_name=ENDPOINT)]:
    w.vector_search_indexes.create_index(
        name=INDEX,
        endpoint_name=ENDPOINT,
        primary_key="comment_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table="zomato_ai.gold.comment_embeddings",
            pipeline_type=PipelineType.TRIGGERED,
            embedding_vector_columns=[EmbeddingVectorColumn(
                name="embedding", embedding_dimension=1024)]))
    print("index requested")
else:
    w.vector_search_indexes.sync_index(INDEX)
    print("index already exists, sync requested")

endpoint already exists
index already exists, sync requested


In [0]:
for _ in range(60):
    status = w.vector_search_indexes.get_index(INDEX).status
    print("ready:", status.ready, "|", status.message)
    if status.ready:
        break
    time.sleep(20)

ready: True | Index is currently online, pipeline update is pending setup of pipeline resources. Check latest status: https://dbc-03718793-91b8.cloud.databricks.com/explore/data/zomato_ai/gold/comment_index


## Retrieval and answering
`retrieve` finds the closest comments to a question, optionally filtered by a column such as sentiment. `ask_reviews` gives them to a chat model, which answers only from them and cites the numbered sources.

In [0]:
import json
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

EMBED_MODEL = "databricks-gte-large-en"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"  # the name that worked in 02_text_to_sql
COLS = ["comment", "sentiment", "topic", "review_count", "avg_rating"]

def embed(text):
    resp = w.serving_endpoints.query(name=EMBED_MODEL, input=[text])
    return resp.data[0].embedding

def retrieve(question, k=30, filters=None):
    res = w.vector_search_indexes.query_index(
        index_name=INDEX, columns=COLS,
        query_vector=embed(question), num_results=k,
        filters_json=json.dumps(filters) if filters else None)
    hits = []
    for row in res.result.data_array:
        d = dict(zip(COLS, row[:len(COLS)]))
        d["score"] = row[-1]
        hits.append(d)
    return hits

def ask_reviews(question, k=30, filters=None):
    hits = retrieve(question, k, filters)
    context = "\n".join(
        f'[{i+1}] "{h["comment"]}" (sentiment: {h["sentiment"]}; topic: {h["topic"]}; '
        f'used in {int(h["review_count"])} reviews; avg rating {h["avg_rating"]})'
        for i, h in enumerate(hits)
    )
    system = ("You answer questions about customer reviews of food delivery orders. "
              "Use ONLY the reviews provided. Cite review numbers like [1]. "
              "The reviews provided are only a small sample of the closest matches, "
              "not all the reviews. Do not conclude that something does not exist in "
              "the data just because it is missing from this sample. "
              "If the sample does not answer the question, say the sample does not cover it. "
              "Do not invent details. 'Used in N reviews' is how many reviews contain that exact comment.")
    resp = w.serving_endpoints.query(
        name=CHAT_MODEL,
        messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=system),
                  ChatMessage(role=ChatMessageRole.USER,
                              content=f"Reviews:\n{context}\n\nQuestion: {question}")],
        max_tokens=500, temperature=0)
    print("Question:", question, "| filters:", filters, "\n")
    print(resp.choices[0].message.content)
    print("\nSources:\n" + context)

In [0]:
for h in retrieve("problems with the delivery partner", k=5):
    print(round(float(h["score"]), 3), "|", h["comment"], "|", h["topic"], "|", h["sentiment"])

0.005 | The delivery partner was rude. | delivery partner behavior | negative
0.004 | The delivery partner was rude. Delivery took over an hour. | delivery partner behavior | negative
0.004 | Delivery took over an hour. The delivery partner was rude. | delivery partner behavior | negative
0.003 | The delivery partner was rude. Arrived way too late. | delivery partner behavior | negative
0.003 | The delivery partner was rude. Quality has really dropped lately. | delivery partner behavior | negative


## Example questions
Without a filter, a question about packaging complaints returns packaging praise, because embeddings capture the topic more strongly than the sentiment. Filtering on sentiment fixes it. A question the data can't answer (parking) should be reported as not covered.

In [0]:
ask_reviews("What do customers complain about regarding packaging?",
            k=30, filters={"sentiment": "negative"})

Question: What do customers complain about regarding packaging? | filters: {'sentiment': 'negative'} 

Customers complain about the packaging leaking everywhere [13, 14, 18, 20] and gravy spilling all over the bag [25]. These issues are mentioned in multiple reviews, with "The packaging leaked everywhere" being used in 3004 reviews [13] and "Gravy spilled all over the bag" being used in 2990 reviews [25].

Sources:
[1] "The delivery partner was rude." (sentiment: negative; topic: delivery partner behavior; used in 2984 reviews; avg rating 1.5)
[2] "The delivery partner was rude. Quality has really dropped lately." (sentiment: negative; topic: delivery partner behavior; used in 193 reviews; avg rating 1.5)
[3] "Delivery took over an hour." (sentiment: negative; topic: delivery speed; used in 2094 reviews; avg rating 1.49)
[4] "The rider got lost twice. The delivery partner was rude." (sentiment: negative; topic: delivery partner behavior; used in 261 reviews; avg rating 1.49)
[5] "The d

In [0]:
ask_reviews("What do customers say about the delivery partners?")

Question: What do customers say about the delivery partners? | filters: None 

Customers have both positive and negative comments about the delivery partners. 

Positive comments include: 
- "The delivery partner was polite and helpful" [used in 9665 reviews, among others].
- This phrase is often combined with other positive comments, such as "Delivered right on time" [4, 7, 24], "Arrived earlier than expected" [6, 16], "Super fast delivery" [13, 21, 26], and "Smooth, hassle-free experience" [24, 26].

Negative comments include: 
- "The delivery partner was rude" [5, 9, 10, 12, 14, 15, 19, 28, 30].
- Some reviews also mention issues with the delivery partner getting lost [9, 10] or arriving late [12, 14, 30]. 

The sample does not cover all possible opinions about delivery partners, but it provides a general idea of the common positive and negative comments.

Sources:
[1] "The delivery partner was polite and helpful." (sentiment: positive; topic: delivery partner behavior; used in 9665

In [0]:
ask_reviews("What do customers say about parking?")

Question: What do customers say about parking? | filters: None 

The sample does not cover it. There is no mention of parking in the provided reviews.

Sources:
[1] "The rider got lost twice. The delivery partner was rude." (sentiment: negative; topic: delivery partner behavior; used in 261 reviews; avg rating 1.49)
[2] "Smooth, hassle-free experience." (sentiment: positive; topic: other; used in 9780 reviews; avg rating 4.62)
[3] "The delivery partner was rude. The rider got lost twice." (sentiment: negative; topic: delivery partner behavior; used in 241 reviews; avg rating 1.48)
[4] "The delivery partner was polite and helpful." (sentiment: positive; topic: delivery partner behavior; used in 9665 reviews; avg rating 4.61)
[5] "The food was absolutely delicious. Smooth, hassle-free experience." (sentiment: positive; topic: food quality; used in 621 reviews; avg rating 4.61)
[6] "The delivery partner was rude." (sentiment: negative; topic: delivery partner behavior; used in 2984 review

## Check against the data
The packaging complaints the model found should match a direct count from the table. "The packaging leaked everywhere" should appear with 3,004 reviews.

In [0]:
%sql
SELECT comment, count(*) AS reviews
FROM zomato_ai.gold.review_enriched
WHERE sentiment = 'negative'
  AND (lower(comment) LIKE '%packag%' OR lower(comment) LIKE '%spill%' OR lower(comment) LIKE '%leak%')
GROUP BY comment
ORDER BY reviews DESC
LIMIT 10

comment,reviews
The packaging leaked everywhere.,3004
Gravy spilled all over the bag.,2990
Overpriced and not worth it. The packaging leaked everywhere.,391
Gravy spilled all over the bag. The delivery partner was rude.,388
The packaging leaked everywhere. Overpriced and not worth it.,387
Gravy spilled all over the bag. Far too expensive for what you get.,385
Far too expensive for what you get. Gravy spilled all over the bag.,383
Got the wrong order entirely. Gravy spilled all over the bag.,380
Got the wrong order entirely. The packaging leaked everywhere.,378
Gravy spilled all over the bag. Got the wrong order entirely.,376
